# Base Dataset Generation: Mass CUDA Ingestion and CoT Labeling

**Notebook Scope:** Core Dataset Construction  
**Objective:** Scale the dataset by ingesting high-performance NVIDIA repositories and generating high-fidelity Chain-of-Thought (CoT) migration notes.  
**Hardware Profile:** 2x NVIDIA T4 GPUs (Batch Processing Optimized)

### Methodology
1. **Massive Ingestion:** Cloning 20+ specialized repositories to capture diverse GPU patterns.
2. **Structural Filtering:** Isolating files with `__global__` and `__device__` tags to ensure 100% GPU-relevant data.
3. **Automated Conversion:** Leveraging `HIPIFY-perl` for the initial CUDA-to-HIP mapping.
4. **LLM Distillation:** Using `Qwen2.5-Coder-7B-Instruct` in a multi-GPU setup to generate architectural and migration reasoning.

## 1. Multi-GPU Inference Strategy
To process over 1,000 kernels efficiently, we utilize `device_map="auto"` to split the 7B model across both T4 GPUs. We implement a custom `DataLoader` with a batch size of 4, optimizing the VRAM usage for high-throughput generation.

## 2. Phase 1: Architectural Profiling
Before migration, the model analyzes the CUDA code to identify:
* **Memory Models:** Use of shared memory, registers, or global coalescing.
* **Parallel Patterns:** Reduction, tiling, or sliding window kernels.
* **Bottlenecks:** Potential latency or occupancy issues in the original code.

In [1]:
!git clone https://github.com/NVIDIA/cuda-samples
!git clone https://github.com/NVIDIA/cutlass
!git clone https://github.com/NVIDIA/thrust
!git clone https://github.com/moderngpu/moderngpu

!git clone https://github.com/ROCm/HIPIFY

Cloning into 'cuda-samples'...
remote: Enumerating objects: 31311, done.
remote: Counting objects: 100% (7061/7061), done.
remote: Compressing objects: 100% (223/223), done.
remote: Total 31311 (delta 6869), reused 6838 (delta 6838), pack-reused 24250 (from 1)
Receiving objects: 100% (31311/31311), 136.93 MiB | 31.26 MiB/s, done.
Resolving deltas: 100% (27580/27580), done.
Updating files: 100% (2054/2054), done.
Cloning into 'cutlass'...
remote: Enumerating objects: 42752, done.
remote: Counting objects: 100% (393/393), done.
remote: Compressing objects: 100% (276/276), done.
remote: Total 42752 (delta 248), reused 117 (delta 117), pack-reused 42359 (from 4)
Receiving objects: 100% (42752/42752), 66.45 MiB | 29.87 MiB/s, done.
Resolving deltas: 100% (32297/32297), done.
Cloning into 'thrust'...
remote: Enumerating objects: 50516, done.
remote: Counting objects: 100% (234/234), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 50516 (delta 98), reused 199 (delta 85)

In [2]:
!git clone https://github.com/NVIDIA/cub
!git clone https://github.com/NVIDIA/apex
!git clone https://github.com/NVIDIA/Megatron-LM
!git clone https://github.com/facebookresearch/xformers
!git clone https://github.com/NVIDIA/cuda-samples

Cloning into 'cub'...
remote: Enumerating objects: 33392, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 33392 (delta 209), reused 184 (delta 184), pack-reused 33145 (from 4)
Receiving objects: 100% (33392/33392), 18.00 MiB | 27.93 MiB/s, done.
Resolving deltas: 100% (27972/27972), done.
Cloning into 'apex'...
remote: Enumerating objects: 13204, done.
remote: Counting objects: 100% (710/710), done.
remote: Compressing objects: 100% (500/500), done.
remote: Total 13204 (delta 406), reused 211 (delta 210), pack-reused 12494 (from 4)
Receiving objects: 100% (13204/13204), 16.69 MiB | 2.52 MiB/s, done.
Resolving deltas: 100% (8930/8930), done.
Cloning into 'Megatron-LM'...
remote: Enumerating objects: 106335, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 106335 (delta 194), reused 132 (delta 126), pack-reused 106068 (from 2)
Receiving objects: 100% 

In [3]:
!git clone https://github.com/NVIDIA/cccl
!git clone https://github.com/NVIDIA/CUDALibrarySamples
!git clone https://github.com/NVIDIA/nvbench
!git clone https://github.com/NVIDIA/warp
!git clone https://github.com/NVIDIA/cuCollections

!git clone https://github.com/ShlokVFX/100-days-cuda
!git clone https://github.com/abaksy/cuda-examples
!git clone https://github.com/drkennetz/cuda_examples
!git clone https://github.com/ashokyannam/GPU_Acceleration_Using_CUDA_C_CPP
!git clone https://github.com/udacity/cs344

!git clone https://github.com/clu0/unet.cu
!git clone https://github.com/mobiusml/gemlite
!git clone https://github.com/Dao-AILab/flash-attention
!git clone https://github.com/NVIDIA/FasterTransformer
!git clone https://github.com/NVIDIA/TransformerEngine

!git clone https://github.com/NVIDIA/amgx
!git clone https://github.com/NVIDIA/cuda-quantum
!git clone https://github.com/rapidsai/cudf
!git clone https://github.com/rapidsai/cuml

Cloning into 'cccl'...
remote: Enumerating objects: 317076, done.
remote: Counting objects: 100% (2437/2437), done.
remote: Compressing objects: 100% (466/466), done.
remote: Total 317076 (delta 2212), reused 1989 (delta 1967), pack-reused 314639 (from 3)
Receiving objects: 100% (317076/317076), 124.96 MiB | 27.51 MiB/s, done.
Resolving deltas: 100% (245311/245311), done.
Cloning into 'CUDALibrarySamples'...
remote: Enumerating objects: 9106, done.
remote: Counting objects: 100% (2641/2641), done.
remote: Compressing objects: 100% (704/704), done.
remote: Total 9106 (delta 2109), reused 2198 (delta 1909), pack-reused 6465 (from 2)
Receiving objects: 100% (9106/9106), 40.04 MiB | 32.93 MiB/s, done.
Resolving deltas: 100% (5992/5992), done.
Cloning into 'nvbench'...
remote: Enumerating objects: 6055, done.
remote: Counting objects: 100% (3270/3270), done.
remote: Compressing objects: 100% (800/800), done.
remote: Total 6055 (delta 2824), reused 2500 (delta 2466), pack-reused 2785 (from 3

In [4]:
import os

cuda_files = []

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        if f.endswith(".cu"):
            cuda_files.append(os.path.join(root,f))

print("Total .cu files:", len(cuda_files))

# mostrar algunos
for f in cuda_files[:10]:
    print(f)

Total .cu files: 5632
/kaggle/working/CUDALibrarySamples/nvCOMP/benchmarks/benchmark_bitcomp_chunked.cu
/kaggle/working/CUDALibrarySamples/nvCOMP/benchmarks/benchmark_lz4_chunked.cu
/kaggle/working/CUDALibrarySamples/nvCOMP/benchmarks/benchmark_gdeflate_chunked.cu
/kaggle/working/CUDALibrarySamples/nvCOMP/benchmarks/benchmark_zstd_chunked.cu
/kaggle/working/CUDALibrarySamples/nvCOMP/benchmarks/benchmark_ans_chunked.cu
/kaggle/working/CUDALibrarySamples/nvCOMP/benchmarks/benchmark_snappy_chunked.cu
/kaggle/working/CUDALibrarySamples/nvCOMP/benchmarks/benchmark_cascaded_chunked.cu
/kaggle/working/CUDALibrarySamples/nvCOMP/benchmarks/benchmark_bitcomp_lossy.cu
/kaggle/working/CUDALibrarySamples/nvCOMP/benchmarks/benchmark_deflate_chunked.cu
/kaggle/working/CUDALibrarySamples/nvCOMP/examples/deflate_cpu_compression.cu


In [5]:
count = 0

for file in cuda_files:

    code = open(file, errors="ignore").read()

    if "__global__" in code:
        print("Kernel encontrado en:", file)
        count += 1

        if count > 10:
            break

print("Archivos con kernels:", count)

Kernel encontrado en: /kaggle/working/CUDALibrarySamples/nvCOMP/examples/nvcomp_gds.cu
Kernel encontrado en: /kaggle/working/CUDALibrarySamples/cuFFT/lto_callback_window_1d/src/r2c_c2r_reference.cu
Kernel encontrado en: /kaggle/working/CUDALibrarySamples/cuPQC/example_sha2.cu
Kernel encontrado en: /kaggle/working/CUDALibrarySamples/cuPQC/example_poseidon2.cu
Kernel encontrado en: /kaggle/working/CUDALibrarySamples/cuPQC/example_ml_dsa.cu
Kernel encontrado en: /kaggle/working/CUDALibrarySamples/cuPQC/example_merkle.cu
Kernel encontrado en: /kaggle/working/CUDALibrarySamples/cuPQC/example_ml_kem.cu
Kernel encontrado en: /kaggle/working/CUDALibrarySamples/cuPQC/example_sha3.cu
Kernel encontrado en: /kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/18_tensor_transform/gemm_conj_transpose.cu
Kernel encontrado en: /kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/18_tensor_transform/trsm_conj_transpose.cu
Kernel encontrado en: /kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/12_gemm_devi

In [6]:
for file in cuda_files:

    code = open(file, errors="ignore").read()

    pos = code.find("__global__")

    if pos != -1:
        print("Archivo con kernel:", file)
        print("\n--- KERNEL PREVIEW ---\n")
        print(code[pos:pos+500])
        break

Archivo con kernel: /kaggle/working/CUDALibrarySamples/nvCOMP/examples/nvcomp_gds.cu

--- KERNEL PREVIEW ---

__global__ void initialize(uint8_t* data, size_t n)
{
  size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n)
    data[i] = i & 0xff;
}

// Kernel to compare 2 buffers. Invalid flag must be set to zero before.
__global__ void
compare(const uint8_t* ref, const uint8_t* val, int* invalid, size_t n)
{
  size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
  int stride = gridDim.x * blockDim.x;
  while (i < n) {
    if (ref[i] != val[i])
      *invalid = 1;
    i += stride;
  }


In [7]:
kernel_files = []
kernel_count = 0

for file in cuda_files:

    code = open(file, errors="ignore").read()

    if "__global__" in code:
        kernel_files.append(file)
        kernel_count += code.count("__global__")

print("Archivos con kernels:", len(kernel_files))
print("Total kernels encontrados:", kernel_count)

print("\nEjemplos de archivos con kernels:")
for f in kernel_files[:10]:
    print(f)

Archivos con kernels: 1133
Total kernels encontrados: 2890

Ejemplos de archivos con kernels:
/kaggle/working/CUDALibrarySamples/nvCOMP/examples/nvcomp_gds.cu
/kaggle/working/CUDALibrarySamples/cuFFT/lto_callback_window_1d/src/r2c_c2r_reference.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_sha2.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_poseidon2.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_ml_dsa.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_merkle.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_ml_kem.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_sha3.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/18_tensor_transform/gemm_conj_transpose.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/18_tensor_transform/trsm_conj_transpose.cu


In [8]:
count = 0
files_with_global = []

for f in cuda_files:
    code = open(f, errors="ignore").read()
    if "__global__" in code:
        count += 1
        files_with_global.append(f)

print("Archivos con __global__:", count)

print("\nEjemplos:")
for f in files_with_global[:10]:
    print(f)

Archivos con __global__: 1133

Ejemplos:
/kaggle/working/CUDALibrarySamples/nvCOMP/examples/nvcomp_gds.cu
/kaggle/working/CUDALibrarySamples/cuFFT/lto_callback_window_1d/src/r2c_c2r_reference.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_sha2.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_poseidon2.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_ml_dsa.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_merkle.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_ml_kem.cu
/kaggle/working/CUDALibrarySamples/cuPQC/example_sha3.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/18_tensor_transform/gemm_conj_transpose.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/18_tensor_transform/trsm_conj_transpose.cu


In [9]:
device_files = []

for f in cuda_files:
    code = open(f, errors="ignore").read()

    if "__device__" in code or "__host__" in code:
        device_files.append(f)

print("Archivos con device/host:", len(device_files))

for f in device_files[:10]:
    print(f)

Archivos con device/host: 1142
/kaggle/working/CUDALibrarySamples/cuFFT/lto_callback_window_1d/src/r2c_c2r_lto_callback_device.cu
/kaggle/working/CUDALibrarySamples/cuFFT/lto_callback_window_1d/src/r2c_c2r_legacy_callback_example.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/07_gemm_transform/simple_gemm_transform.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/13_gemm_fft/gemm_fft.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/13_gemm_fft/gemm_fft_performance.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuBLASDx/13_gemm_fft/gemm_fft_fp16.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuFFTDx/06_convolution/convolution_performance.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuFFTDx/04_nvrtc_fft/nvrtc_fft_block.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuFFTDx/04_nvrtc_fft/nvrtc_query_database_fft_block.cu
/kaggle/working/CUDALibrarySamples/MathDx/cuFFTDx/04_nvrtc_fft/nvrtc_fft_block_lto.cu


In [10]:
import re

kernel_pattern = re.compile(
    r"__global__\s+void\s+[a-zA-Z0-9_]+\s*\([^)]*\)\s*\{",
    re.MULTILINE
)

def extract_kernels(code):

    kernels = []

    for match in kernel_pattern.finditer(code):

        start = match.start()

        brace = 0
        i = match.end() - 1

        while i < len(code):

            if code[i] == "{":
                brace += 1

            elif code[i] == "}":
                brace -= 1

                if brace == 0:
                    kernels.append(code[start:i+1])
                    break

            i += 1

    return kernels

In [11]:
total = 0

for f in cuda_files[:100]:

    code = open(f, errors="ignore").read()

    ks = extract_kernels(code)

    if ks:
        print("archivo:", f)
        print("kernels:", len(ks))
        print("\n--- ejemplo ---\n")
        print(ks[0][:400])
        print("\n----------------\n")

    total += len(ks)

print("kernels encontrados en prueba:", total)

archivo: /kaggle/working/CUDALibrarySamples/nvCOMP/examples/nvcomp_gds.cu
kernels: 2

--- ejemplo ---

__global__ void initialize(uint8_t* data, size_t n)
{
  size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n)
    data[i] = i & 0xff;
}

----------------

kernels encontrados en prueba: 2


In [12]:
all_kernels = []

for f in cuda_files:

    code = open(f, errors="ignore").read()

    ks = extract_kernels(code)

    all_kernels.extend(ks)

print("Total kernels extraídos:", len(all_kernels))

Total kernels extraídos: 2448


In [13]:
import pandas as pd

df = pd.DataFrame({
    "kernel": all_kernels
})

df.to_csv("cuda_kernels_dataset.csv", index=False)

print("Dataset guardado")
print("Total kernels:", len(df))

Dataset guardado
Total kernels: 2448


In [14]:
# ============================================
# CUDA → HIP DATASET GENERATION (HIPIFY)
# ============================================
import os
import subprocess

hip_pairs = []

def run_hipify(cuda_code):
    """
    Usa la herramienta hipify-perl incluida en ROCm/HIPIFY 
    para convertir CUDA → HIP directamente.
    """
    with open("temp.cu", "w") as f:
        f.write(cuda_code)

    # Llamamos a perl y le pasamos la ruta correcta del script de HIPIFY
    result = subprocess.run(
        ["perl", "HIPIFY/bin/hipify-perl", "temp.cu"],
        capture_output=True,
        text=True
    )

    return result.stdout if result.stdout else None

for f in cuda_files:  # Limitado a 500 para evitar agotar recursos en Kaggle
    try:
        code = open(f, errors="ignore").read()

        # Solo convertimos si detectamos un kernel
        if "__global__" in code:

            hip_code = run_hipify(code)

            if hip_code:
                hip_pairs.append({
                    "input": code,
                    "output": hip_code,
                    "source": f
                })

    except Exception as e:
        print(f"No se pudo procesar {f}: {e}")
        continue

print("Pares CUDA→HIP generados:", len(hip_pairs))

Pares CUDA→HIP generados: 1133


In [15]:
import os
import subprocess
import json
from tqdm import tqdm

hip_pairs = []
archivos_procesados = 0
archivos_con_error = 0

print("Iniciando conversión de CUDA a HIP...")

# Creamos o abrimos el archivo en modo escritura (sobrescribe si existe)
with open("cuda_to_hip_dataset.jsonl", "w") as out_file:
    # Recorremos todos los archivos sin el límite [:500]
    for f in tqdm(cuda_files):
        try:
            code = open(f, errors="ignore").read()

            if "__global__" in code:
                # Archivo temporal para HIPIFY
                with open("temp.cu", "w") as temp_f:
                    temp_f.write(code)

                # Ejecutamos hipify-perl
                result = subprocess.run(
                    ["perl", "HIPIFY/bin/hipify-perl", "temp.cu"],
                    capture_output=True,
                    text=True
                )

                if result.stdout:
                    pair_data = {
                        "input": code,
                        "output": result.stdout,
                        "source": f
                    }
                    # Guardamos inmediatamente en disco para no perder datos
                    out_file.write(json.dumps(pair_data) + "\n")
                    archivos_procesados += 1

        except Exception as e:
            archivos_con_error += 1
            continue

print(f"Proceso terminado. Pares generados y guardados: {archivos_procesados}")
print(f"Errores de lectura/conversión: {archivos_con_error}")

Iniciando conversión de CUDA a HIP...


100%|██████████| 5632/5632 [02:07<00:00, 44.30it/s] 

Proceso terminado. Pares generados y guardados: 1133
Errores de lectura/conversión: 0


In [16]:
# ============================================
# UPLOAD TO HUGGING FACE (SAFE TOKEN HANDLING)
# ============================================

from huggingface_hub import login, HfApi
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")


login(token=HF_TOKEN)

api = HfApi()

repo_id = "Daleth-hb/cuda-hip-gpu-dataset"

api.create_repo(repo_id=repo_id, exist_ok=True)

api.upload_file(
    path_or_fileobj="cuda_to_hip_dataset.jsonl",
    path_in_repo="cuda_to_hip_dataset.jsonl",
    repo_id=repo_id
)

print("Dataset subido a Hugging Face:", repo_id)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Dataset subido a Hugging Face: Daleth-hb/cuda-hip-gpu-dataset


In [17]:
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00


## 3. Phase 2: Translation & Reasoning (CoT)
We map the `input` (CUDA) to the `output` (HIP) and use the LLM to generate a **Migration Notes** section. This provides the "labels" that the LoRA adapter will eventually learn to mimic, focusing on API parity and architectural differences between CUDA and ROCm.

In [18]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

print("Configurando Qwen2.5 para 2xT4 GPUs...")
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"

# 1. Tokenizador configurado para batching (Padding a la izquierda)
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

# 2. device_map="auto" dividirá el modelo mágicamente entre cuda:0 y cuda:1
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Detectamos dónde está la capa de entrada para mandar los datos ahí
input_device = next(model.parameters()).device 

# 3. Preparar los archivos
kernel_files = [f for f in cuda_files if "__global__" in open(f, errors="ignore").read()]
print(f"Archivos con kernels a analizar: {len(kernel_files)}")

# 4. Clase Dataset para agrupar los prompts
class CudaArchDataset(Dataset):
    def __init__(self, files):
        self.files = files
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        f = self.files[idx]
        code = open(f, errors="ignore").read()
        # Truncamos a ~4000 caracteres para no volar la VRAM en los batches
        if len(code) > 4000: code = code[:4000] + "\n// [TRUNCADO]"
        
        prompt = f"Explain this CUDA kernel at an architectural level.\nFocus on: 1. Memory model 2. Parallel pattern 3. Optimizations 4. Bottlenecks\nCode:\n{code}"
        
        messages = [
            {"role": "system", "content": "You are an expert GPU architecture engineer. Be extremely concise and technical."},
            {"role": "user", "content": prompt}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return {"text": text, "code": code, "source": f}

# BATCH_SIZE = 4 es el punto dulce para dos T4 con un modelo de 7B. 
# Si te da error de memoria, bájalo a 2.
BATCH_SIZE = 4
loader = DataLoader(CudaArchDataset(kernel_files), batch_size=BATCH_SIZE)

print("Iniciando generación arquitectural por lotes...")

with open("cuda_arch_dataset.jsonl", "w") as out_file:
    for batch in tqdm(loader):
        try:
            # Tokenizamos todo el batch junto
            inputs = tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True, max_length=1500)
            
            # Movemos los datos a la GPU correcta (cuda:0)
            inputs = {k: v.to(input_device) for k, v in inputs.items()}

            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=300,
                    pad_token_id=tokenizer.eos_token_id,
                    do_sample=False # Más rápido
                )
            
            # Cortamos el prompt original para guardar solo la respuesta
            prompt_lengths = inputs["input_ids"].shape[1]
            responses = tokenizer.batch_decode(generated_ids[:, prompt_lengths:], skip_special_tokens=True)

            # Guardamos cada elemento del batch
            for i in range(len(responses)):
                record = {
                    "instruction": "Analyze the following CUDA kernel architecture.",
                    "input": batch["code"][i],
                    "output": responses[i],
                    "source": batch["source"][i]
                }
                out_file.write(json.dumps(record) + "\n")
            
        except Exception as e:
            print(f"Error en batch (OOM o similar). Saltando... Error: {e}")
            continue

print("Dataset arquitectural completado.")

Configurando Qwen2.5 para 2xT4 GPUs...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Archivos con kernels a analizar: 1133
Iniciando generación arquitectural por lotes...


  0%|          | 0/284 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Dataset arquitectural completado.


In [19]:
import json
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import HfApi
from tqdm.auto import tqdm

print("Cargando pares CUDA-HIP previamente generados...")
hip_pairs = []
with open("cuda_to_hip_dataset.jsonl", "r") as f:
    for line in f:
        hip_pairs.append(json.loads(line))

# Para el hackathon, limitaremos a 500 para tener un dataset MVP de alta calidad hoy mismo
#hip_pairs = hip_pairs[:500] 

class CoTDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        cuda_code = pair["input"][:2500]
        hip_code = pair["output"][:2500]
        
        prompt = f"You are migrating CUDA to AMD ROCm (HIP). Explain the changes step-by-step for this translation. Mention API mappings and architectural notes.\n[CUDA]\n{cuda_code}\n[HIP]\n{hip_code}"
        
        messages = [
            {"role": "system", "content": "You are a senior GPU software engineer. Be highly technical, precise, and concise."},
            {"role": "user", "content": prompt}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return {"text": text, "cuda": pair["input"], "hip": pair["output"], "source": pair.get("source", "unknown")}

cot_loader = DataLoader(CoTDataset(hip_pairs), batch_size=BATCH_SIZE)

print("Generando explicaciones de migración (CoT) por lotes...")

with open("cuda_hip_cot_dataset.jsonl", "w") as out_file:
    for batch in tqdm(cot_loader):
        try:
            inputs = tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True, max_length=2000)
            inputs = {k: v.to(input_device) for k, v in inputs.items()}

            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=300,
                    pad_token_id=tokenizer.eos_token_id,
                    do_sample=False
                )
            
            prompt_lengths = inputs["input_ids"].shape[1]
            explanations = tokenizer.batch_decode(generated_ids[:, prompt_lengths:], skip_special_tokens=True)

            for i in range(len(explanations)):
                record = {
                    "instruction": "Migrate this CUDA kernel to AMD ROCm (HIP) and explain the reasoning step-by-step.",
                    "input": batch["cuda"][i],
                    # ESTO FUNCIONA PERFECTO
                    "output": f"```cpp\n{batch['hip'][i]}\n```\n\n### Migration Notes:\n{explanations[i]}",
                    "source": batch["source"][i]
                }
                out_file.write(json.dumps(record) + "\n")
                
        except Exception as e:
            print(f"Error en batch CoT: {e}")
            continue

print("Dataset de Razonamiento completado.")

# Subida a Hugging Face
print("Subiendo a Hugging Face...")
api = HfApi()
repo_id = "Daleth-hb/cuda-hip-gpu-dataset"
api.upload_file(
    path_or_fileobj="cuda_hip_cot_dataset.jsonl",
    path_in_repo="cuda_hip_cot_dataset.jsonl",
    repo_id=repo_id
)
print(f"¡Éxito! Dataset subido a: https://huggingface.co/datasets/{repo_id}")

Cargando pares CUDA-HIP previamente generados...
Generando explicaciones de migración (CoT) por lotes...


  0%|          | 0/284 [00:00<?, ?it/s]

Dataset de Razonamiento completado.
Subiendo a Hugging Face...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

¡Éxito! Dataset subido a: https://huggingface.co/datasets/Daleth-hb/cuda-hip-gpu-dataset
